In [1]:
from gpt import GPTModel, load_weights_into_gpt, Config
from safetensors.torch import load_file
import matplotlib.pyplot as plt
from typing import Literal
import torch.nn as nn
import requests
import tiktoken
import torch
import os

## Download the weights of the model

In [2]:
if not os.path.exists(os.path.join("data", "model.safetensors")):
    os.system("hf download openai-community/gpt2 model.safetensors --local-dir ./data")

In [3]:
# config = {
#     "vocab_size": 50257,    # Vocabulary size
#     "context_length": 1024, # Context length
#     "drop_rate": 0.0,       # Dropout rate
#     "qkv_bias": True,        # Query-key-value bias
#     "emb_dim": 768,
#     "n_layers": 12,
#     "n_heads": 12,
# }

config = Config()

In [4]:
model = GPTModel(config)
model

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [5]:
state_dict = load_file("./data/model.safetensors")

In [6]:
load_weights_into_gpt(model, state_dict)

/home/shillianne/Documents/AI/NLP/P1/NLPP1/gpt.py:188: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.nn.Parameter(torch.tensor(right))


In [10]:
print(f"Number of parameters: {sum(p.nelement() for p in model.parameters()):,}")

Number of parameters: 163,037,184


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [8]:
model.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [9]:
encoding = tiktoken.get_encoding("gpt2")
encoding

<Encoding 'gpt2'>

In [10]:
def generate(
    model: GPTModel,
    text: str,
    n: int,
    behavior: Literal["best", "temp"] = "best",
    temp: float = 0.0,
    topk: int = 1,
    topp: float = 0.1,
    device: str = "cuda",
):
    print(f"Input: {text}")
    print(f"Response: ")
    x = torch.tensor(encoding.encode(text)).to(device)
    for _ in range(n):
        with torch.inference_mode():
            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                y = model(x.unsqueeze(0))
        if behavior == "best":
            choice = torch.argmax(y[:, -1, :].squeeze(0), dim=-1, keepdim=True)
        else:  # temp
            assert temp > 0.0, "For a temperature based decoding, please ensure the temperature is higher than 0"
            y = (y[:, -1, :] / temp).squeeze(0)
            indices = torch.topk(y, topk).indices
            keep = torch.zeros_like(y, device=y.device, dtype=torch.bool)
            keep[indices] = True
            y = torch.clone(y)
            y[torch.logical_not(y)] = float("-inf")

            sorted_logits, sorted_indices = torch.sort(y, descending=True)
            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            remove = cumulative_probs > topp
            remove[1:] = remove[:-1].clone()
            remove[0] = False
            y[sorted_indices[remove]] = float("-inf")
            choice = torch.multinomial(torch.softmax(y, dim=-1), num_samples=1)

        x = torch.hstack([x, choice])
        token = encoding.decode([choice.item()])
        print(f"{token}", end="")

In [11]:
generate(
    model,
    "Hello there, how are you?",
    10,
    behavior="best",
    temp=0.3,
    topk=2,
    topp=0.8
)

Input: Hello there, how are you?
Response: 


I'm a little bit of a geek

## Using LatentMoE

In [12]:
from src.moe.latent_moe import LatentMoe

model = GPTModel(config, ff=LatentMoe)
model

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): LatentMoe(
        (W_down): Linear(in_features=768, out_features=256, bias=False)
        (W_up): Linear(in_features=256, out_features=768, bias=False)
        (experts): ModuleList(
          (0-7): 8 x Expert(
            (w1): Linear(in_features=256, out_features=256, bias=False)
            (w2): Linear(in_features=256, out_features=256, bias=False)
            (w3): Linear(in_features=256, out_features=256, bias=False)
    

In [13]:
load_weights_into_gpt(model, state_dict, latent=True)

In [14]:
# model.to("cuda")

In [16]:
generate(
    model,
    "How are you?",
    10,
    behavior="best",
    temp=0.3,
    topk=2,
    topp=0.8,
    device = "cpu"
)

Input: How are you?
Response: 
iusesiusesiusesiusesiusesiusesiusesiusesiusesiuses